# Car Detection in Snow — Training Notebook
**Group 10 | D7047E Advanced Deep Learning**

This notebook trains both models:
- **V1**: YOLOv9 baseline (standard augmentation)
- **V2**: YOLOv9 + snow-aware augmentation

Run cells top to bottom. Each section is clearly labelled.

## 0. Setup — run once

In [1]:
import os
os.chdir('/project')   # set working directory to project root


In [2]:
# Install dependencies if not already installed
# (on LTU cluster these should already be available)
import subprocess
subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)  # installs this project
subprocess.run(['pip', 'install', 'omegaconf', 'albumentations', 'wandb', 'optuna', '-q'], check=True)
print('Dependencies ready.')

Dependencies ready.


In [3]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.11.0+cu130
GPU available: True
GPU: NVIDIA GeForce RTX 2080 Ti


## 1. Load config

In [4]:
from src.common_utils import load_config
from src.common_utils.seed import set_seed_from_config
from src.data import DataPipeline
from src.model.baseline_yolov9.trainer import YOLOv9Trainer
from src.model.snow_augmented_yolov9.trainer_sa import YOLOv9TrainerSA
from src.evaluation.evaluate import Evaluator

print("All modules imported successfully!")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All modules imported successfully!


In [5]:
from src.common_utils import load_config

# Load V1 config to inspect it
cfg_v1 = load_config(variant='v1')

print('=== V1 Config ===')
print(f'  Experiment : {cfg_v1.experiment}')
print(f'  Epochs     : {cfg_v1.training.epochs}')
print(f'  LR         : {cfg_v1.training.lr}')
print(f'  Batch size : {cfg_v1.training.batch_size}')
print(f'  Snow aug   : {cfg_v1.use_snow_aug}')
print(f'  Device     : {cfg_v1.project.device}')

=== V1 Config ===
  Experiment : v1_baseline
  Epochs     : 100
  LR         : 0.001
  Batch size : 16
  Snow aug   : False
  Device     : cuda:0


## 2. Check the dataset
This runs DataPipeline . it will process the NVD videos on first run (slow), then skip on future runs.

In [12]:
from src.data import DataPipeline

pipeline = DataPipeline('config.yaml')
pipeline.summary()   # prints train/val/test image and bbox counts

Setting up YOLO dataset from NVD...
  Processing 2022-12-04 Bjenberg 02 → train


FileNotFoundError: [Errno 2] No such file or directory: 'src/data/NVD'

In [ ]:
# Visual check — see 3 random training images with bounding boxes
# Change augment to 'snow' to preview snow transforms
pipeline.show_samples(n=3, split='train', augment='base')

In [ ]:
# Preview snow augmentation
pipeline.show_samples(n=3, split='train', augment='snow')

In [6]:
#delete when data pipeline is ready
from src.common_utils import load_config
from src.model.baseline_yolov9.trainer import _ensure_yolov9
from pathlib import Path

cfg = load_config(variant="v1")
_ensure_yolov9(Path(cfg.model.yolov9_repo))
print("YOLOv9 cloned and ready!")

YOLOv9 cloned and ready!


## 3. Train V1 — Baseline

**Note:** If hyperparams are already tuned the best hyperparams are already written into `config.yaml`. load config updates the hyperparams no manual changes needed.

In [7]:
from src.model.baseline_yolov9.trainer import YOLOv9Trainer
from src.common_utils import load_config

cfg_v1    = load_config(variant='v1')
trainer_v1 = YOLOv9Trainer(cfg_v1)

# This runs the full training loop
# Saves checkpoints to outputs/checkpoints/v1/
# Best model saved as outputs/checkpoints/v1/best.pt
trainer_v1.train()

/project/third_party/yolov9/utils/general.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


Setting up YOLO dataset from NVD...
  Processing 2022-12-04 Bjenberg 02 → train


FileNotFoundError: [Errno 2] No such file or directory: 'src/data/NVD'

In [ ]:
# Quick validation check after training
v1_metrics = trainer_v1.validate()
print('\n=== V1 Final Metrics ===')
print(f"  mAP50    : {v1_metrics['mAP50']:.4f}")
print(f"  mAP50-95 : {v1_metrics['mAP50_95']:.4f}")

## 4. Train V2 — Snow Augmented

In [ ]:
from src.model.snow_augmented_yolov9.trainer_sa import YOLOv9TrainerSA

cfg_v2     = load_config(variant='v2')
trainer_v2 = YOLOv9TrainerSA(cfg_v2)

# Saves to outputs/checkpoints/v2/best.pt
trainer_v2.train()

In [ ]:
v2_metrics = trainer_v2.validate()
print('\n=== V2 Final Metrics ===')
print(f"  mAP50    : {v2_metrics['mAP50']:.4f}")
print(f"  mAP50-95 : {v2_metrics['mAP50_95']:.4f}")

## 5. Compare V1 vs V2

In [ ]:
from src.common_utils.checkpoint import get_best_metric

v1 = get_best_metric('outputs/checkpoints/v1/best.pt')
v2 = get_best_metric('outputs/checkpoints/v2/best.pt')

print('\n========== Results Summary ==========')
print(f"{'Metric':<15} {'V1 Baseline':>12} {'V2 Snow-Aug':>12} {'Improvement':>12}")
print('-' * 55)
for key in ['mAP50', 'mAP50_95']:
    v1_val = v1.get(key, 0.0)
    v2_val = v2.get(key, 0.0)
    diff   = v2_val - v1_val
    sign   = '+' if diff >= 0 else ''
    print(f"{key:<15} {v1_val:>12.4f} {v2_val:>12.4f} {sign+f'{diff:.4f}':>12}")

winner = 'V2 Snow-Augmented' if v2.get('mAP50', 0) > v1.get('mAP50', 0) else 'V1 Baseline'
print(f'\nBetter model: {winner}')

## 6. Done!

Checkpoints are saved in `outputs/checkpoints/`.
Hand off `best.pt` paths to the evaluation person (TBD).

```
outputs/checkpoints/v1/best.pt  ← V1 baseline
outputs/checkpoints/v2/best.pt  ← V2 snow-augmented
```